In [209]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage,SystemMessage
import os
from dotenv import load_dotenv
import operator

In [210]:
load_dotenv()

True

In [211]:
api_key = os.getenv("GROQ_API")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing from .env")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    groq_api_key=api_key
)

In [212]:
generate_llm = llm
evaluate_llm = llm
optimize_llm = llm

In [213]:
class evaluateSchema(BaseModel):
    feedback: str = Field(...,desc="feedback of the tweet")
    evaluation: Literal["approved","needs_improvement"] = Field(...,desc="Final Evaluation of the tweet")

C:\Users\hashr\AppData\Local\Temp\ipykernel_13168\1509036402.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desc'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  feedback: str = Field(...,desc="feedback of the tweet")
C:\Users\hashr\AppData\Local\Temp\ipykernel_13168\1509036402.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desc'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  evaluation: Literal["approved","needs_improvement"] = Field(...,desc="Final Evaluation of the tweet")


In [214]:
structured_model = evaluate_llm.with_structured_output(evaluateSchema)

In [215]:
class tweetState(TypedDict):
    tweet: str
    evaluate: Literal["approved","need_improvement"]
    feedback: str
    topic: str
    iteration: int
    max_iteration: int 
    tweet_history: Annotated[list[str],operator.add]
    feedback_history: Annotated[list[str],operator.add]

In [216]:
def generate_tweet(state: tweetState):
    messages=[
        SystemMessage(content="You are hularious and clever Twitter/X influencer"),
        HumanMessage(content=f"""
        Write a short, Original and influenced tweet on the topic: "{state["topic"]}"

        Follow these rules:

        Write exactly 5 lines.

        Keep the language simple, natural, and impactful.

        Make it creative and attention-grabbing.

        Include a fresh perspective or unique insight.

        Avoid repetition, clichés, and unnecessary hashtags .

        """)
    ]

    response = generate_llm.invoke(messages).content

    return {'tweet': response, 'tweet_history': [response]}


In [217]:
def evaluate_tweet(state: tweetState):
    messages = [
        SystemMessage(
      content="""
      You are an expert social media content evaluator.
  
      Evaluate the given tweet based on these criteria:
      1. Originality and creativity.
      2. Relevance to the given topic.
      3. Engagement and audience appeal.
      4. Clarity and readability.
      5. Compliance with the exactly 5-line rule.
  
      Give a score out of 10.
      Provide a short explanation for your score.
      Suggest improvements if necessary.
  
      Return your evaluation in a clear and structured format.
     """
  ),HumanMessage(
    content=f"""
    Topic: {state["topic"]}

    Generated Tweet:
    {state["tweet"]}

    Evaluate this tweet according to the given criteria.
    """
)]

    response = structured_model.invoke(messages)

    return {'evaluate':response.evaluation,'feedback':response.feedback, 'feedback_history': [response.feedback]}
    

In [218]:
def optimize_tweet(state: tweetState):
    messages=[
        SystemMessage(
    content="""
    You are an expert Twitter/X content optimizer.

    Your task is to improve the given tweet based on the
    evaluator's feedback.

    Follow these rules:
    1. Write exactly 5 lines.
    2. Make the tweet original, creative, and engaging.
    3. Improve the opening hook to attract attention.
    4. Keep the tweet relevant to the given topic.
    5. Use simple, natural, and impactful language.
    6. Remove unnecessary words and repetition.
    7. Incorporate the evaluator's suggestions.
    8. Preserve the original meaning and key message.

    Return only the optimized tweet.
    Do not include explanations, scores, or headings.
    """
),
HumanMessageHumanMessage(
    content=f"""
    Topic: {state["topic"]}

    Original Tweet:
    {state["tweet"]}

    Evaluator Feedback:
    {state["feedback"]}

    Optimize the original tweet according to the evaluator's
    feedback.

    Ensure the optimized tweet contains exactly 5 lines.
    """
)]

    response = optimize_llm.invoke(messages).content
    iteration = state["iteration"]+1

    return {'tweet':response, 'iteration':iteration, 'tweet_history': [response]}

In [219]:
def route_evaluation(state: tweetState):
    if state["evaluate"] == 'approved' or max_iteration >=5:
        return 'approved'
    else:
        return 'need_improvement'  

In [220]:
graph = StateGraph(tweetState)
graph.add_node("generate",generate_tweet)
graph.add_node("evaluate",evaluate_tweet)
graph.add_node("optimize",optimize_tweet)

graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')
graph.add_conditional_edges('evaluate',route_evaluation,{'approved':END,'needs_evaluation':'optimize'})
graph.add_edge('optimize','evaluate')


In [225]:
workflow = graph.compile()
initial_state={
    "topic":"fen",
    'max_iteration':5,
    "iteration":0

}
workflow.invoke(initial_state)

{'tweet': 'When the sun dips, the fen turns into a secret library of whispers.  \nEvery ripple writes a story in the mud, unheard by the city.  \nIf you listen, you’ll hear the ancient songs of reeds and frogs.  \nIt’s not just water; it’s a living archive of resilience.  \nStep into the fen and discover the quiet roar of nature’s archive.',
 'evaluate': 'approved',
 'feedback': 'Score: 9/10\n\nOriginality & Creativity: The tweet paints a vivid, poetic picture of a fen, using metaphors like a secret library and living archive. It stands out from typical nature posts.\n\nRelevance: The content is tightly focused on a fen, describing its sounds, sights, and ecological significance.\n\nEngagement: The evocative language invites readers to imagine and explore, likely sparking curiosity and shares.\n\nClarity & Readability: Sentences are concise, easy to read, and each line flows naturally.\n\n5‑Line Rule: Exactly five lines are present, meeting the structural requirement.\n\nSuggestions: C